# KPIs para Dashboard

## Célula 1 — Configuração

In [37]:
from pyspark.sql import functions as F

GOLD_PATH = "s3a://gold/tse"

print(f"Gold Path: {GOLD_PATH}")

Gold Path: s3a://gold/tse


## Célula 2 — Carregar tabelas Gold

In [38]:
fato_candidatura_dashboard = spark.read.format("delta").load(
    f"{GOLD_PATH}/fato_candidatura_dashboard"
)

fato_bem_candidato_dashboard = spark.read.format("delta").load(
    f"{GOLD_PATH}/fato_bem_candidato_dashboard"
)

dim_partido = spark.read.format("delta").load(
    f"{GOLD_PATH}/dim_partido"
)

dim_municipio = spark.read.format("delta").load(
    f"{GOLD_PATH}/dim_municipio"
)

## KPI 1 — Total de Candidatos

In [39]:
from pyspark.sql import functions as F

kpi_total_candidatos = (
    fato_candidatura_dashboard
    .agg(
        F.countDistinct("sq_candidato")
        .alias("total_candidatos")
    )
    .withColumn("data_referencia", F.current_timestamp())
)

kpi_total_candidatos.show()

+----------------+--------------------+
|total_candidatos|     data_referencia|
+----------------+--------------------+
|           78349|2026-06-23 02:57:...|
+----------------+--------------------+



In [40]:
kpi_total_candidatos.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}/kpi_total_candidatos")

## KPI 2 — Total de Partidos

In [41]:
kpi_total_partidos = (
    dim_partido
    .agg(
        F.countDistinct("numero_partido")
        .alias("total_partidos")
    )
    .withColumn("data_referencia", F.current_timestamp())
)

kpi_total_partidos.show()

+--------------+--------------------+
|total_partidos|     data_referencia|
+--------------+--------------------+
|            29|2026-06-23 02:57:...|
+--------------+--------------------+



In [42]:
kpi_total_partidos.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}/kpi_total_partidos")

## KPI 3 — Total de Municípios

In [43]:
kpi_total_municipios = (
    dim_municipio
    .agg(
        F.count("*")
        .alias("total_municipios")
    )
    .withColumn("data_referencia", F.current_timestamp())
)

kpi_total_municipios.show()

+----------------+--------------------+
|total_municipios|     data_referencia|
+----------------+--------------------+
|             645|2026-06-23 02:57:...|
+----------------+--------------------+



In [44]:
kpi_total_municipios.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}/kpi_total_municipios")

## KPI 4 — Bens Declarados

In [45]:
total_bens = (
    fato_bem_candidato_dashboard
    .agg(
        F.sum("valor_bem_candidato")
        .alias("valor_total_bens")
    )
)

In [46]:
media_bens = (
    fato_bem_candidato_dashboard
    .groupBy("sq_candidato")
    .agg(
        F.sum("valor_bem_candidato")
        .alias("total_bens_candidato")
    )
    .agg(
        F.avg("total_bens_candidato")
        .alias("media_bens_por_candidato")
    )
)

## Partido com maior patrimônio

In [47]:
patrimonio_partido = (
    fato_bem_candidato_dashboard
    .join(
        fato_candidatura_dashboard,
        "sq_candidato"
    )
    .groupBy("numero_partido")
    .agg(
        F.sum("valor_bem_candidato")
        .alias("valor_partido")
    )
)

In [48]:
partido_maior = (
    patrimonio_partido
    .join(
        dim_partido,
        "numero_partido"
    )
    .orderBy(F.desc("valor_partido"))
    .limit(1)
)

## Montagem final do KPI

In [49]:
kpi_bens_declarados = (
    total_bens
    .crossJoin(media_bens)
    .crossJoin(
        partido_maior.select(
            F.col("sigla_partido")
                .alias("partido_maior_patrimonio"),
            F.col("valor_partido")
                .alias("valor_partido_maior_patrimonio")
        )
    )
    .withColumn(
        "data_referencia",
        F.current_timestamp()
    )
)

kpi_bens_declarados.show(truncate=False)

+--------------------+------------------------+------------------------+------------------------------+--------------------------+
|valor_total_bens    |media_bens_por_candidato|partido_maior_patrimonio|valor_partido_maior_patrimonio|data_referencia           |
+--------------------+------------------------+------------------------+------------------------------+--------------------------+
|2.041006283881002E10|414047.6090154987       |PRTB                    |3.24973252258E9               |2026-06-23 02:57:30.691505|
+--------------------+------------------------+------------------------+------------------------------+--------------------------+



In [50]:
kpi_bens_declarados.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}/kpi_bens_declarados")

## Validação 

In [51]:
spark.read.format("delta").load(
    f"{GOLD_PATH}/kpi_total_candidatos"
).show()

+----------------+--------------------+
|total_candidatos|     data_referencia|
+----------------+--------------------+
|           78349|2026-06-23 02:57:...|
+----------------+--------------------+



In [52]:
spark.read.format("delta").load(
    f"{GOLD_PATH}/kpi_total_partidos"
).show()

+--------------+--------------------+
|total_partidos|     data_referencia|
+--------------+--------------------+
|            29|2026-06-23 02:57:...|
+--------------+--------------------+



In [53]:
spark.read.format("delta").load(
    f"{GOLD_PATH}/kpi_total_municipios"
).show()

+----------------+--------------------+
|total_municipios|     data_referencia|
+----------------+--------------------+
|             645|2026-06-23 02:57:...|
+----------------+--------------------+



In [54]:
spark.read.format("delta").load(
    f"{GOLD_PATH}/kpi_bens_declarados"
).show(truncate=False)

+--------------------+------------------------+------------------------+------------------------------+--------------------------+
|valor_total_bens    |media_bens_por_candidato|partido_maior_patrimonio|valor_partido_maior_patrimonio|data_referencia           |
+--------------------+------------------------+------------------------+------------------------------+--------------------------+
|2.041006283881002E10|414047.6090154987       |PRTB                    |3.24973252258E9               |2026-06-23 02:57:31.631082|
+--------------------+------------------------+------------------------+------------------------------+--------------------------+

